# Cameroon Laws FAISS Index Builder

This notebook builds a FAISS vector index from 54 Cameroon law PDFs.

**Steps:**
1. Download PDFs from GitHub automatically
2. Run all cells
3. Download the generated index files

**Output files:**
- `index_file.index` - FAISS vector index
- `index_file.meta.json` - Article metadata
- `index_file.chunks.json` - Article text chunks

In [ ]:
# Install dependencies
!pip install -q faiss-cpu==1.7.4 sentence-transformers==2.2.2 PyPDF2==3.0.1 openai==1.3.0 numpy==1.24.3

In [ ]:
# Download PDFs from GitHub automatically
import os
import urllib.request

print("Downloading PDFs from GitHub...")
print("This may take a few minutes...\n")

# Clone the repository
!git clone https://github.com/A-Njock/cameroon-laws-rag.git

# Count PDF files
pdf_files = [f for f in os.listdir('cameroon-laws-rag') if f.endswith('.pdf')]
print(f"\n✓ Downloaded {len(pdf_files)} PDF files")

# Move PDFs to current directory
for pdf in pdf_files:
    !mv "cameroon-laws-rag/{pdf}" .

print("✓ PDFs ready for processing")

In [ ]:
# Download RAG.py from GitHub
!wget -q https://raw.githubusercontent.com/A-Njock/cameroon-laws-rag/main/RAG.py
print("✓ Downloaded RAG.py")

In [ ]:
# Build FAISS index
from RAG import build_faiss_from_folder
import os

# Current directory contains downloaded PDFs
folder_path = "."
index_path = "index_file.index"
metadata_path = "index_file.meta.json"

print("="*60)
print("Building FAISS Index")
print("="*60)
print(f"Processing PDFs from: {folder_path}")
print(f"This will take ~10-15 minutes...\n")

build_faiss_from_folder(
    folder_path=folder_path,
    index_output_path=index_path,
    metadata_output_path=metadata_path,
    embedding_model='all-MiniLM-L6-v2'
)

print("\n" + "="*60)
print("✓ Index building complete!")
print("="*60)

In [ ]:
# Verify index files
import os

files_to_check = [
    "index_file.index",
    "index_file.meta.json",
    "index_file.chunks.json"
]

print("Index files created:")
total_size = 0
for f in files_to_check:
    if os.path.exists(f):
        size = os.path.getsize(f) / (1024 * 1024)  # MB
        total_size += size
        print(f"  ✓ {f} ({size:.2f} MB)")
    else:
        print(f"  ✗ {f} - NOT FOUND")

print(f"\nTotal size: {total_size:.2f} MB")

In [ ]:
# Download index files
from google.colab import files
import os

print("Downloading index files...")

files_to_download = [
    "index_file.index",
    "index_file.meta.json",
    "index_file.chunks.json"
]

for f in files_to_download:
    if os.path.exists(f):
        files.download(f)
        print(f"  ✓ Downloaded {f}")

print("\n✓ All files downloaded!")
print("\nNext steps:")
print("1. Upload these files to Cloudflare R2")
print("2. Deploy the API to Cloudflare Workers")